# NLP + AI Chatbot Crop Recommendation System

In [1]:
!pip install pandas numpy nltk scikit-learn

In [2]:

import pandas as pd
import numpy as np
import nltk
import re

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:

df = pd.read_csv('crop_recommendation.csv')
df.head()


,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


In [4]:

encoder = LabelEncoder()
df['label'] = encoder.fit_transform(df['label'])

X = df[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']]
y = df['label']


In [5]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

print("Model trained successfully")


Model trained successfully


In [6]:

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z0-9 ]', '', text)

    tokens = word_tokenize(text)

    processed_tokens = []

    for token in tokens:

        if token not in stop_words:

            processed_tokens.append(
                lemmatizer.lemmatize(token)
            )

    return processed_tokens


In [7]:

soil_types = ['sandy', 'clay', 'loam']

climate_conditions = [
    'hot',
    'cold',
    'humid',
    'dry'
]

rainfall_conditions = [
    'low rainfall',
    'high rainfall',
    'moderate rainfall'
]


In [8]:

def extract_features(query):

    query = query.lower()

    extracted = {
        'soil': None,
        'climate': None,
        'rainfall': None
    }

    for soil in soil_types:

        if soil in query:
            extracted['soil'] = soil

    for climate in climate_conditions:

        if climate in query:
            extracted['climate'] = climate

    for rainfall in rainfall_conditions:

        if rainfall in query:
            extracted['rainfall'] = rainfall

    return extracted


In [9]:

def generate_feature_vector(extracted):

    N = 70
    P = 40
    K = 40
    temperature = 25
    humidity = 60
    ph = 6.5
    rainfall = 100

    if extracted['soil'] == 'sandy':
        humidity = 40
        rainfall = 50

    elif extracted['soil'] == 'clay':
        humidity = 75
        rainfall = 120

    elif extracted['soil'] == 'loam':
        humidity = 60
        rainfall = 100

    if extracted['climate'] == 'hot':
        temperature = 35

    elif extracted['climate'] == 'cold':
        temperature = 18

    elif extracted['climate'] == 'humid':
        humidity = 85

    elif extracted['climate'] == 'dry':
        humidity = 30

    if extracted['rainfall'] == 'low rainfall':
        rainfall = 40

    elif extracted['rainfall'] == 'high rainfall':
        rainfall = 200

    elif extracted['rainfall'] == 'moderate rainfall':
        rainfall = 100

    return [[
        N,
        P,
        K,
        temperature,
        humidity,
        ph,
        rainfall
    ]]


In [10]:

def recommend_top_3_crops(query):

    cleaned_tokens = preprocess_text(query)

    print('Processed Tokens:', cleaned_tokens)

    extracted = extract_features(query)

    print('Extracted Features:', extracted)

    feature_vector = generate_feature_vector(extracted)

    probabilities = model.predict_proba(feature_vector)[0]

    top_3_indices = np.argsort(probabilities)[-3:][::-1]

    recommendations = []

    for idx in top_3_indices:

        crop_name = encoder.inverse_transform([idx])[0]

        confidence = probabilities[idx] * 100

        recommendations.append({
            'crop': crop_name,
            'confidence': round(confidence, 2)
        })

    return recommendations, extracted


In [11]:

def generate_explanations(extracted):

    explanations = []

    if extracted['soil']:
        explanations.append(
            f"Suitable for {extracted['soil']} soil"
        )

    if extracted['climate']:
        explanations.append(
            f"Thrives in {extracted['climate']} conditions"
        )

    if extracted['rainfall']:
        explanations.append(
            f"Performs well under {extracted['rainfall']}"
        )

    return explanations


# Advanced Recommendation Logic

In [12]:

def recommend_crops(features):

    probabilities = model.predict_proba([features])[0]

    top_3 = np.argsort(probabilities)[-3:][::-1]

    recommendations = []

    for index in top_3:

        crop = encoder.inverse_transform([index])[0]

        confidence = probabilities[index] * 100

        recommendations.append({
            'crop': crop,
            'confidence': round(confidence, 2)
        })

    return recommendations


In [13]:

def calculate_weighted_score(
    probability,
    temperature,
    rainfall
):

    score = probability

    if temperature > 30:
        score += 5

    if rainfall < 60:
        score += 5

    return score


In [14]:

def weighted_recommendation(features):

    probabilities = model.predict_proba([features])[0]

    recommendations = []

    for idx, prob in enumerate(probabilities):

        score = calculate_weighted_score(
            prob * 100,
            features[3],
            features[6]
        )

        crop = encoder.inverse_transform([idx])[0]

        recommendations.append({
            'crop': crop,
            'score': score
        })

    recommendations = sorted(
        recommendations,
        key=lambda x: x['score'],
        reverse=True
    )

    return recommendations[:3]


In [15]:
def preprocess_text(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z0-9 ]', '', text)

    # Simple tokenizer
    tokens = text.split()

    processed_tokens = []

    for token in tokens:

        if token not in stop_words:

            processed_tokens.append(
                lemmatizer.lemmatize(token)
            )

    return processed_tokens

In [16]:

user_query = 'Which crop should I grow in sandy soil with low rainfall?'

recommendations, extracted = recommend_top_3_crops(user_query)

explanations = generate_explanations(extracted)

print('\nTop 3 Recommended Crops:\n')

for i, rec in enumerate(recommendations, start=1):

    print(f"{i}. {rec['crop']} — {rec['confidence']}% suitability")

print('\nReasons:\n')

for reason in explanations:
    print('-', reason)


Processed Tokens: ['crop', 'grow', 'sandy', 'soil', 'low', 'rainfall']
Extracted Features: {'soil': 'sandy', 'climate': None, 'rainfall': 'low rainfall'}

Top 3 Recommended Crops:

1. coffee — 25.0% suitability
2. mothbeans — 17.5% suitability
3. maize — 15.0% suitability

Reasons:

- Suitable for sandy soil
- Performs well under low rainfall


c:\Users\USER\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [17]:
query = input('Farmer Query: ')

recommendations, extracted = recommend_top_3_crops(query)

explanations = generate_explanations(extracted)

print('\nTop Recommendations:\n')

for i, rec in enumerate(recommendations, start=1):

    print(f"{i}. {rec['crop']} — {rec['confidence']}% suitability")

print('\nReasoning:')

for reason in explanations:
    print('-', reason)

Processed Tokens: []
Extracted Features: {'soil': None, 'climate': None, 'rainfall': None}

Top Recommendations:

1. maize — 35.0% suitability
2. coffee — 32.5% suitability
3. jute — 11.5% suitability

Reasoning:


c:\Users\USER\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
